In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install -q datasets huggingface_hub pillow tqdm

In [ ]:
from huggingface_hub import login

login()

In [ ]:
CLASS_MAPPING = {
    "airplane": "airliner",
    "automobile": "sports car",
    "bird": "goldfinch",
    "cat": "tabby",
    "deer": "red deer",
    "dog": "golden retriever",
    "frog": "tree frog",
    "horse": "sorrel",
    "ship": "container ship",
    "truck": "trailer truck"
}

SPLIT_CONFIG = {
    "train": 1000,
    "val": 200,
    "test": 200
}

SAVE_DIR = "/content/drive/MyDrive/ImageNet10"

In [ ]:
import os
from PIL import Image
from tqdm.auto import tqdm

from collections import defaultdict

# Tạo thư mục
for split in SPLIT_CONFIG:
    for class_name in class_to_id:
        os.makedirs(
            f"{SAVE_DIR}/{split}/{class_name}",
            exist_ok=True
        )

# Tổng ảnh cần lấy mỗi class
TOTAL_PER_CLASS = sum(SPLIT_CONFIG.values())

counts = defaultdict(int)

id_to_class = {
    label_id: class_name
    for class_name, label_id in class_to_id.items()
}

total_needed = len(class_to_id) * TOTAL_PER_CLASS

pbar = tqdm(total=total_needed)

for sample in dataset:

    label_id = sample[label_key]

    if label_id not in id_to_class:
        continue

    class_name = id_to_class[label_id]

    if counts[class_name] >= TOTAL_PER_CLASS:
        continue

    image = sample["image"].convert("RGB")

    index = counts[class_name]

    # Xác định split
    if index < SPLIT_CONFIG["train"]:
        split = "train"

    elif index < SPLIT_CONFIG["train"] + SPLIT_CONFIG["val"]:
        split = "val"

    else:
        split = "test"

    image_path = (
        f"{SAVE_DIR}/{split}/{class_name}/"
        f"{index:05d}.jpg"
    )

    image.save(image_path, quality=95)

    counts[class_name] += 1
    pbar.update(1)

    if all(
        counts[c] >= TOTAL_PER_CLASS
        for c in class_to_id
    ):
        break

pbar.close()

print("Download completed!")
print(dict(counts))

In [ ]:
for split, expected in SPLIT_CONFIG.items():

    print(f"\n{split.upper()}")

    for class_name in class_to_id:

        folder = f"{SAVE_DIR}/{split}/{class_name}"

        n_images = len([
            f for f in os.listdir(folder)
            if f.endswith(".jpg")
        ])

        print(f"{class_name}: {n_images}")